# 🚀 Оптимизация PySpark: PostgreSQL, join, shuffle, перекос данных и S3

**Вебинар Simulative · WEB-03**

Каждый пример собран по одному шаблону:

| Ячейка | Что внутри |
|---|---|
| ❌ **Как НЕ надо** | типичная ошибка, которую делают в проде |
| ✅ **Как надо** | исправление и объяснение, почему оно работает |
| 📊 **Метрики** | цифры из Spark REST API (то же, что в Spark UI) + подсказки, где смотреть в UI и что «плохо» |

### Стенд

```
                ┌──────────────────── сервер 45.144.220.99 ──────────────────────┐
  IDE / браузер ┼──► Jupyter :8888 ──► PySpark 4.1 (local[8], driver 6 GB)       │
  браузер ──────┼──► Spark UI :4040          │                    │              │
  браузер ──────┼──► History Server :18080   │ JDBC               │ S3A          │
                │                   ┌────────▼────────┐           │              │
  DBeaver ──────┼──► :5432 ─────────┤ PostgreSQL 17   │           │              │
                │                   │ база shop       │           │              │
                │                   └─────────────────┘           │              │
                └─────────────────────────────────────────────────┼──────────────┘
                                                                  ▼
                                        S3 Timeweb: <bucket>/simulative/web-03/
```

### Данные (в них специально заложены проблемы)

| Таблица | Строк | Роль в демо |
|---|---|---|
| `countries` | 200 | крошечный справочник → broadcast |
| `products` | 50 000 | маленький справочник → broadcast |
| `customers` | 2 000 000 | большая таблица → join через shuffle |
| `orders` | 10 000 000 | факт-таблица с **перекосом**: 60% заказов у `customer_id = 1` (оптовый партнёр), 15% с `customer_id IS NULL` (гостевые), 25% равномерно |

### План

0. Подготовка: SparkSession и вспомогательные функции
1. Spark UI и метрики: куда смотреть
2. PostgreSQL: чтение, pushdown, запись
3. Читаем план: `explain()`, Exchange, SortMergeJoin, BroadcastHashJoin
4. Оптимизация shuffle
5. Перекос данных (data skew)
6. S3: форматы, партиционирование, мелкие файлы, committer
7. Итоги: таблица «было → стало» и чеклист

---
# 0. Подготовка

Все настройки Spark собраны в одной ячейке и прокомментированы.

> ⚠️ `spark.driver.memory` применяется **только при старте JVM**. Если Spark в этом kernel уже запускался, сделайте *Restart Kernel* и выполните ячейку заново: иначе настройка молча проигнорируется.

Пароли и ключи в ноутбуке не хранятся: они приходят из переменных окружения контейнера (файл `/opt/spark-demo/.env` на сервере).

In [ ]:
# ⚙️ SparkSession: все настройки в одном месте
import os, io, re, json, time, uuid, urllib.request
from contextlib import contextmanager, redirect_stdout

import pandas as pd
from IPython.display import display, Markdown
from pyspark.sql import SparkSession, Window, functions as F

PUBLIC_HOST = os.environ["PUBLIC_HOST"]
S3_ROOT = f"s3a://{os.environ['S3_BUCKET']}/{os.environ['S3_PREFIX']}"
PG_URL = f"jdbc:postgresql://{os.environ['PG_HOST']}:{os.environ['PG_PORT']}/{os.environ['POSTGRES_DB']}"
PG_USER, PG_PASSWORD = os.environ["POSTGRES_USER"], os.environ["POSTGRES_PASSWORD"]

spark = (
    SparkSession.builder
    .appName("webinar-pyspark-optimization")
    .master("local[8]")                          # 8 ядер сервера = до 8 задач параллельно
    .config("spark.driver.memory", "6g")         # в local mode драйвер одновременно и executor
    .config("spark.ui.port", "4040")
    .config("spark.ui.showConsoleProgress", "false")
    # История запусков для Spark History Server (:18080): UI переживёт перезапуск kernel
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:/home/jovyan/spark-events")
    .config("spark.sql.warehouse.dir", "/home/jovyan/spark-warehouse")
    .config("spark.sql.adaptive.enabled", "true")  # AQE включён по умолчанию, в демо будем его выключать
    .config("spark.sql.maxMetadataStringLength", "1000")  # explain() не обрезает PushedFilters на 100 символах

    # ---------- S3: коннектор S3A (hadoop-aws) ----------
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])  # https://s3.twcstorage.ru
    .config("spark.hadoop.fs.s3a.endpoint.region", "ru-1")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["S3_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["S3_SECRET_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")  # https://host/bucket/key: нужно большинству S3-совместимых хранилищ
    .config("spark.hadoop.fs.s3a.connection.maximum", "200")  # пул HTTP-соединений (по умолчанию мало для параллельного чтения)
    .config("spark.hadoop.fs.s3a.threads.max", "64")          # потоки для загрузки и листинга
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version, "| master:", sc.master, "| app:", sc.applicationId)

In [ ]:
# 🧰 Вспомогательные функции демо. Запустите и сверните: на вебинаре разбирать не обязательно.
#   measure(label)        — замер времени + пометка всех Spark-джобов внутри блока
#   show_metrics(*labels) — метрики stages/SQL из Spark REST API + автоматическая диагностика
#   explain_highlights()  — explain() с расшифровкой: где shuffle, broadcast, pushdown
#   final_plan()          — план ПОСЛЕ выполнения (с учётом решений AQE)
#   set_conf / reset_conf — меняем настройки на время демо и возвращаем значения по умолчанию
#   s3_stats / s3_delete  — сколько файлов и какого размера лежит по пути в S3

UI_INTERNAL = sc.uiWebUrl                       # адрес UI изнутри контейнера (для REST API)
UI_PORT = UI_INTERNAL.rsplit(":", 1)[1]
UI_PUBLIC = f"http://{PUBLIC_HOST}:{UI_PORT}"    # адрес UI для вашего браузера
APP_ID = sc.applicationId
RUNS = {}
_TOUCHED_CONF = set()


def set_conf(conf):
    for key, value in conf.items():
        spark.conf.set(key, str(value))
        _TOUCHED_CONF.add(key)
        print(f"⚙️  {key} = {value}")


def reset_conf():
    for key in _TOUCHED_CONF:
        spark.conf.unset(key)
    _TOUCHED_CONF.clear()


def _fs(path):
    jpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    return jpath.getFileSystem(sc._jsc.hadoopConfiguration()), jpath


# Счётчики коннектора S3A: реальные байты по сети и HTTP-запросы к S3
_S3_COUNTERS = {
    "stream_read_bytes": "прочитано",
    "object_put_bytes": "записано",
    "object_list_request": "LIST",
    "action_http_head_request": "HEAD",
    "action_http_get_request": "GET",
    "object_put_request": "PUT",
    "object_copy_requests": "COPY",
    "object_delete_request": "DELETE",
    "object_bulk_delete_request": "BULK DELETE",
}


def _s3_counters():
    it = _fs(S3_ROOT)[0].getStorageStatistics().getLongStatistics()
    out = {}
    while it.hasNext():
        st = it.next()
        if st.getName() in _S3_COUNTERS:
            out[st.getName()] = st.getValue()
    return out


@contextmanager
def measure(label, case=None, variant=None):
    group = f"demo-{uuid.uuid4().hex[:8]}"
    s3_before = _s3_counters()
    sc.setJobGroup(group, label)
    t0 = time.perf_counter()
    try:
        yield
    finally:
        wall = time.perf_counter() - t0
        sc.setJobGroup("adhoc", "вне замеров")
        s3_after = _s3_counters()
        s3 = {k: s3_after[k] - s3_before.get(k, 0) for k in _S3_COUNTERS
              if s3_after.get(k, 0) - s3_before.get(k, 0) > 0}
        RUNS[label] = {"group": group, "wall": wall, "case": case, "variant": variant, "s3": s3}
        print(f"⏱  {label}: {wall:.1f} c")


# ---------- форматирование ----------
def _bytes(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024:
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"


def _num(n):
    return f"{int(n):,}".replace(",", " ")


def _ts(value):
    return pd.Timestamp(value.replace("GMT", ""))


def _size(text):
    num, _, unit = text.replace(",", "").partition(" ")
    return float(num) * {"B": 1, "KiB": 1024, "MiB": 1024 ** 2, "GiB": 1024 ** 3, "TiB": 1024 ** 4}.get(unit, 1)


def _files_read(nodes):
    files = sum(int(n["файлов прочитано"].replace(",", "")) for n in nodes if "файлов прочитано" in n)
    size = sum(_size(n["объём файлов"]) for n in nodes if "объём файлов" in n)
    return files, size


# ---------- Spark REST API: то же самое, что показывает Spark UI ----------
def _api(path):
    with urllib.request.urlopen(f"{UI_INTERNAL}/api/v1/applications/{APP_ID}/{path}", timeout=120) as resp:
        return json.load(resp)


def _jobs(label):
    return [j for j in _api("jobs") if j.get("jobGroup") == RUNS[label]["group"]]


def _stages(label):
    out = []
    for sid in sorted({s for job in _jobs(label) for s in job["stageIds"]}):
        for st in _api(f"stages/{sid}?details=false"):
            if st["status"] != "COMPLETE":          # SKIPPED = результат shuffle переиспользован
                continue
            q = _api(f"stages/{sid}/{st['attemptId']}/taskSummary?quantiles=0,0.5,1")
            t_min, t_med, t_max = (x / 1000 for x in q["duration"])
            recs = (q["shuffleReadMetrics"]["readRecords"] if st["shuffleReadRecords"]
                    else q["inputMetrics"]["recordsRead"])
            out.append({
                "id": sid, "tasks": st["numTasks"], "start": _ts(st["submissionTime"]),
                "end": _ts(st["completionTime"]), "t_min": t_min, "t_med": t_med, "t_max": t_max,
                "input": st["inputBytes"], "input_rec": st["inputRecords"],
                "sh_read": st["shuffleReadBytes"], "sh_write": st["shuffleWriteBytes"],
                "spill_mem": st["memoryBytesSpilled"], "spill_disk": st["diskBytesSpilled"],
                "gc": st["jvmGcTime"] / 1000, "run": st["executorRunTime"] / 1000, "recs": recs,
            })
    return out


_SQL_METRICS = {
    "number of output rows": "строк на выходе",
    "number of files read": "файлов прочитано",
    "size of files read": "объём файлов",
    "metadata time": "время метаданных",
    "number of partitions read": "папок-партиций прочитано",
    "shuffle records written": "shuffle: записей",
    "shuffle bytes written": "shuffle: записано",
    "data size": "размер данных",
    "number of partitions": "партиций",
    "number of coalesced partitions": "AQE: склеено партиций",
    "number of skewed partitions": "AQE: перекошенных партиций",
    "number of skewed partition splits": "AQE: частей после разбиения",
    "spill size": "spill",
    "time to broadcast": "время broadcast",
}
_SQL_NODES = ("Scan", "Exchange", "BroadcastExchange", "AQEShuffleRead", "SortMergeJoin",
              "BroadcastHashJoin", "ShuffledHashJoin", "Window", "Sort")


def _short(value):
    value = value.split("\n")[1] if "\n" in value else value
    return value.split(" (")[0].strip()


def _sql_nodes(label):
    job_ids = {j["jobId"] for j in _jobs(label)}
    rows = []
    for ex in _api("sql?details=true&planDescription=false&offset=0&length=100000"):
        ids = set(ex.get("successJobIds", []) + ex.get("runningJobIds", []) + ex.get("failedJobIds", []))
        if not ids & job_ids:
            continue
        for node in ex["nodes"]:
            if not node["nodeName"].startswith(_SQL_NODES):
                continue
            metrics = {}
            for m in node.get("metrics", []):
                if m["name"] in _SQL_METRICS:
                    v = _short(m["value"])
                    if m["name"] == "spill size" and v.startswith("0.0"):
                        continue
                    metrics[_SQL_METRICS[m["name"]]] = v
            if metrics:
                rows.append({"sql": ex["id"], "оператор": node["nodeName"][:60], **metrics})
    return rows


def _diagnose(label, stages, nodes):
    run, tips = RUNS[label], []
    if stages:
        span = (max(s["end"] for s in stages) - min(s["start"] for s in stages)).total_seconds()
        outside = run["wall"] - span
        if outside > 3 and outside > 0.25 * run["wall"]:
            tips.append(f"🕐 {outside:.1f} c из {run['wall']:.1f} c прошли ВНЕ задач Spark: работа драйвера "
                        f"(листинг файлов в S3, commit/rename, вывод схемы, планирование). "
                        f"Видно как пустые промежутки в Jobs → Event Timeline")
    for s in stages:
        if s["tasks"] == 1 and s["t_max"] > 3:
            tips.append(f"🐢 Stage {s['id']}: всю работу сделала 1 задача ({s['t_max']:.1f} c), остальные ядра простаивали")
        r_med, r_max = s["recs"][1], s["recs"][2]
        if r_med == 0 and s["tasks"] > 1:
            pass  # больше половины задач пустые (например, после partitionBy/pruning) — это не перекос ключа
        elif s["tasks"] > 1 and s["t_max"] > 1 and s["t_max"] > 4 * max(s["t_med"], 0.05):
            tips.append(f"⚖️  Stage {s['id']}: ПЕРЕКОС. Самая долгая задача {s['t_max']:.1f} c при медиане "
                        f"{s['t_med']:.2f} c (в {s['t_max'] / max(s['t_med'], 0.01):.0f} раз). "
                        f"Stages → stage {s['id']} → Summary Metrics / Event Timeline")
        elif s["tasks"] > 1 and r_max >= 100_000 and r_max > 4 * max(r_med, 1):
            tips.append(f"⚖️  Stage {s['id']}: ПЕРЕКОС по записям. Одна задача обработала {_num(r_max)} записей "
                        f"при медиане {_num(r_med)} (в {r_max / max(r_med, 1):.0f} раз)")
        if s["spill_disk"] > 0:
            tips.append(f"💾 Stage {s['id']}: spill на диск {_bytes(s['spill_disk'])}: задачам не хватило памяти")
        if s["tasks"] >= 500 and s["t_med"] < 0.1:
            tips.append(f"🐜 Stage {s['id']}: {s['tasks']} крошечных задач (медиана {s['t_med'] * 1000:.0f} мс): "
                        f"накладные расходы больше полезной работы")
        if s["run"] > 0 and s["gc"] > 2 and s["gc"] > 0.1 * s["run"]:
            tips.append(f"🧹 Stage {s['id']}: GC занял {s['gc']:.1f} c ({100 * s['gc'] / s['run']:.0f}% времени задач): давление на память")
    files, size = _files_read(nodes)
    if files >= 500:
        tips.append(f"📁 Прочитано {_num(files)} файлов (в среднем {_bytes(size / files)}): мелкие файлы, "
                    f"на каждый файл отдельные HTTP-запросы к S3")
    s3 = run.get("s3", {})
    if s3.get("object_copy_requests"):
        tips.append(f"📋 {_num(s3['object_copy_requests'])} COPY-запросов к S3: это rename при commit "
                    f"(FileOutputCommitter). В S3 rename = копирование + удаление каждого файла")
    requests = sum(v for k, v in s3.items() if not k.endswith("bytes"))
    if requests >= 1000:
        tips.append(f"🌐 {_num(requests)} HTTP-запросов к S3 за один запуск: каждый запрос — это задержка в десятки мс")
    exchanges = [n for n in nodes if n["оператор"].startswith("Exchange")]
    if exchanges:
        tips.append(f"🔀 Shuffle (Exchange): {len(exchanges)} шт., записано {_bytes(sum(s['sh_write'] for s in stages))}")
    if any(n["оператор"].startswith("BroadcastHashJoin") for n in nodes):
        tips.append("📡 BroadcastHashJoin: большая таблица не перемешивалась")
    for n in nodes:
        if n.get("AQE: перекошенных партиций", "0") not in ("0", ""):
            tips.append(f"🩹 AQE нашёл перекошенных партиций: {n['AQE: перекошенных партиций']}, "
                        f"разбил на частей: {n.get('AQE: частей после разбиения', '?')}")
    return tips or ["✅ Явных проблем по метрикам не видно"]


def show_metrics(*labels):
    """Метрики замеров: stages (как вкладка Stages), операторы плана (как вкладка SQL) и диагностика."""
    time.sleep(1.5)  # Spark обновляет статистику асинхронно
    summary = []
    for label in labels:
        if label not in RUNS:
            print(f"⚠️  Нет замера «{label}»: сначала выполните ячейку с ним")
            continue
        stages, nodes = _stages(label), _sql_nodes(label)
        run = RUNS[label]
        display(Markdown(f"#### {label}\n⏱ **{run['wall']:.1f} c** · stages: {len(stages)}"))
        if stages:
            view = pd.DataFrame([{
                "stage": f"stage {s['id']}",
                "задач": s["tasks"],
                "длительность, c": round((s["end"] - s["start"]).total_seconds(), 2),
                "input, записей": _num(s["input_rec"]),
                "shuffle read": _bytes(s["sh_read"]),
                "shuffle write": _bytes(s["sh_write"]),
                "spill память / диск": f"{_bytes(s['spill_mem'])} / {_bytes(s['spill_disk'])}",
                "GC, c": round(s["gc"], 2),
                "задача min / med / max, c": f"{s['t_min']:.2f} / {s['t_med']:.2f} / {s['t_max']:.2f}",
                "записей на задачу min / med / max": " / ".join(_num(r) for r in s["recs"]),
            } for s in stages]).set_index("stage").T
            display(view)
        if nodes:
            display(pd.DataFrame(nodes).fillna("").set_index("sql"))
        s3 = run.get("s3", {})
        if s3:
            print("☁️  S3 (счётчики S3A): " + " · ".join(
                f"{_S3_COUNTERS[k]} {_bytes(v) if k.endswith('bytes') else _num(v)}" for k, v in s3.items()))
        for tip in _diagnose(label, stages, nodes):
            print(tip)
        if stages:
            slow = max(stages, key=lambda s: s["t_max"])
            print(f"🔗 Самый долгий stage в UI: {UI_PUBLIC}/stages/stage/?id={slow['id']}&attempt=0")
        if nodes:
            print(f"🔗 План с метриками (SQL):  {UI_PUBLIC}/SQL/execution/?id={nodes[0]['sql']}")
        files, _ = _files_read(nodes)
        summary.append({
            "запуск": label,
            "время, c": round(run["wall"], 1),
            "задач": sum(s["tasks"] for s in stages),
            "input, записей": _num(sum(s["input_rec"] for s in stages)),
            "S3 прочитано": _bytes(s3.get("stream_read_bytes", 0)) if s3 else "",
            "S3 запросов": _num(sum(v for k, v in s3.items() if not k.endswith("bytes"))) if s3 else "",
            "shuffle write": _bytes(sum(s["sh_write"] for s in stages)),
            "spill диск": _bytes(sum(s["spill_disk"] for s in stages)),
            "макс. задача, c": round(max((s["t_max"] for s in stages), default=0), 2),
            "файлов прочитано": files or "",
        })
    if len(summary) > 1:
        display(Markdown("#### Сравнение"))
        display(pd.DataFrame(summary).set_index("запуск"))


# ---------- план запроса ----------
_PLAN_OPERATORS = [
    (r"Exchange hashpartitioning\([^)]*\)", "🔴 SHUFFLE по ключу (join/groupBy): запись на диск + передача по сети"),
    (r"Exchange rangepartitioning\([^)]*\)", "🔴 SHUFFLE для глобальной сортировки (orderBy)"),
    (r"Exchange RoundRobinPartitioning\(\d+\)", "🟠 SHUFFLE от repartition(n)"),
    (r"Exchange SinglePartition", "🟠 SHUFFLE в ОДНУ партицию: норма для глобального agg из частичных итогов, 🔴 для больших данных"),
    (r"BroadcastExchange \w+", "🟢 BROADCAST: маленькая таблица копируется на каждый executor"),
    (r"BroadcastHashJoin [^\n]*?(?=,|\n)", "🟢 join БЕЗ shuffle большой таблицы"),
    (r"SortMergeJoin(?:\(skew=true\))? [^\n]*?(?=,|\n)", "🟠 join через shuffle + сортировку ОБЕИХ сторон"),
    (r"ShuffledHashJoin [^\n]*?(?=,|\n)", "🟠 join через shuffle (hash, без сортировки)"),
    (r"BroadcastNestedLoopJoin", "🔴 nested loop join: нет условия равенства, очень дорого"),
    (r"Window \[", "🟠 оконная функция: shuffle по partitionBy + сортировка"),
    (r"PushedFilters: \[[^\]]*\]", "🔎 фильтры, переданные в источник ([] = источник отдаёт ВСЁ)"),
    (r"PartitionFilters: \[[^\]]*\]", "🔎 partition pruning ([] = читаются ВСЕ папки-партиции)"),
    (r"ReadSchema: struct<[^>]*>", "🔎 какие колонки реально читаются"),
    (r"isFinalPlan=\w+", "ℹ️  AQE: false — начальный план (может поменяться), true — финальный, после выполнения"),
]


def _highlight(plan):
    print("\n🔎 НА ЧТО СМОТРЕТЬ В ЭТОМ ПЛАНЕ:")
    found = False
    for pattern, meaning in _PLAN_OPERATORS:
        hits = re.findall(pattern, plan)
        if not hits:
            continue
        found = True
        print(f"  {meaning}  × {len(hits)}")
        for hit in dict.fromkeys(hits):
            print(f"       └─ {hit[:150]}")
    if not found:
        print("  нет ни shuffle, ни join")


def explain_highlights(df, mode="formatted", show_plan=True):
    """Печатает df.explain(mode) и ниже расшифровку ключевых операторов."""
    buf = io.StringIO()
    with redirect_stdout(buf):
        df.explain(mode)
    if show_plan:
        print(buf.getvalue())
    simple = io.StringIO()
    with redirect_stdout(simple):
        df.explain("simple")
    _highlight(simple.getvalue())


def final_plan(df):
    """План ПОСЛЕ выполнения (сначала выполните действие над этим же df, например collect())."""
    plan = df._jdf.queryExecution().executedPlan().toString()
    final = plan.split("== Initial Plan ==")[0]
    print(final)
    _highlight(final)


# ---------- S3 ----------
def s3_stats(path):
    fs, jpath = _fs(path)
    files, size, dirs = 0, 0, set()
    it = fs.listFiles(jpath, True)
    while it.hasNext():
        st = it.next()
        if st.getPath().getName().startswith(("_", ".")):
            continue
        files += 1
        size += st.getLen()
        dirs.add(st.getPath().getParent().toString())
    return {"путь": path.replace(S3_ROOT, "…"), "файлов": files, "папок": len(dirs),
            "объём": _bytes(size), "средний файл": _bytes(size / files if files else 0)}


def s3_delete(path):
    fs, jpath = _fs(path)
    fs.delete(jpath, True)


print("Помощники загружены ✅")

In [ ]:
# 🔗 Ссылки стенда: откройте Spark UI в соседней вкладке браузера
print(f"Spark UI:        {UI_PUBLIC}")
print(f"History Server:  http://{PUBLIC_HOST}:18080   (все прошлые запуски, переживает рестарт kernel)")
print(f"S3 root:         {S3_ROOT}")
print(f"PostgreSQL:      {PG_URL}")

---
# 1. Spark UI и метрики: куда смотреть

### Как запрос превращается в задачи

```
 Action (write / count / collect)
  └─ Job
      ├─ Stage 0  — чтение + всё, что можно сделать без перемешивания (filter, select, withColumn)
      │     8 задач (task) = 8 партиций, по одной на ядро
      │     └─ Shuffle Write: каждая задача раскладывает свои строки по ключу на диск
      │
      │  ═══════════ Exchange (shuffle) = граница stage ═══════════
      │
      └─ Stage 1  — join / groupBy / window после перемешивания
            N задач = spark.sql.shuffle.partitions (по умолчанию 200, AQE подстраивает)
            └─ Shuffle Read: каждая задача забирает «свой» ключ со ВСЕХ задач stage 0
```

### Карта Spark UI

```
Spark UI → http://45.144.220.99:4040
┌──────┬────────┬─────────┬─────────────┬───────────┬───────────────┐
│ Jobs │ Stages │ Storage │ Environment │ Executors │ SQL/DataFrame │
└──┬───┴───┬────┴────┬────┴──────┬──────┴─────┬─────┴───────┬───────┘
   │       │         │           │            │             └─ ⭐ граф плана с метриками на каждом операторе:
   │       │         │           │            │                Exchange (сколько перемешали), Scan (сколько файлов),
   │       │         │           │            │                Join (тип), AQEShuffleRead (что сделал AQE)
   │       │         │           │            └─ память, GC, shuffle по executor'ам
   │       │         │           └─ фактические настройки: проверить, что ваш конфиг применился
   │       │         └─ что закешировано и сколько памяти занимает
   │       └─ ⭐ главная вкладка для оптимизации: клик по stage → Summary Metrics, Event Timeline, Tasks
   └─ список действий и их длительность; Event Timeline показывает пустые промежутки (работа драйвера)
```

### Таблица «что плохо»

| Метрика | Где в UI | Плохой признак | Что это значит |
|---|---|---|---|
| Duration: Min / Median / **Max** | Stages → stage → Summary Metrics | Max ≫ Median (в 5–10+ раз) | **перекос данных** |
| **Spill** (Memory / Disk) | Stages, SQL (`spill size`) | больше 0 | задаче не хватило памяти, пишет на диск |
| Shuffle Write / Read | Stages, SQL (Exchange) | сравнимо с размером входа или больше | лишний shuffle, не отфильтровали колонки или строки |
| **Tasks** | Stages | 1 задача на большой объём, или тысячи задач по миллисекунды | нет параллелизма, или слишком мелкие партиции |
| GC Time | Stages, Executors | больше 10% времени задачи | давление на память |
| number of files read | SQL → Scan | тысячи файлов, средний меньше 10 МБ | мелкие файлы |
| Пустоты между jobs | Jobs → Event Timeline | секунды без задач | драйвер листит S3, делает commit/rename |

### Ячейка «📊 Метрики»

`show_metrics()` берёт данные из **Spark REST API**, то есть ровно то же, что показывает Spark UI, и выводит:
1. **Таблицу stages**: задачи, длительность, input, shuffle, spill, GC и разброс времени задач min / med / max.
2. **Операторы плана** с SQL-метриками: файлы, shuffle, решения AQE.
3. **Автодиагностику** (🐢 ⚖️ 💾 🐜 📁 🔀) и **ссылки** прямо на нужный stage в Spark UI.

> 💡 `df.write.format("noop")` выполняет запрос полностью, но никуда не пишет. Это лучший способ замерить скорость: `count()` может «схитрить», потому что Spark выбросит ненужные колонки и даже чтение.

In [ ]:
# Разогрев и знакомство с метриками: groupBy на 20 млн синтетических строк
demo = (spark.range(0, 20_000_000, numPartitions=8)
        .withColumn("key", F.col("id") % 1000)
        .groupBy("key")
        .agg(F.count("*").alias("cnt"), F.sum("id").alias("total")))

with measure("Разогрев: groupBy 20 млн строк"):
    demo.write.format("noop").mode("overwrite").save()

In [ ]:
# 📊 МЕТРИКИ: как читать вывод show_metrics
show_metrics("Разогрев: groupBy 20 млн строк")

**Что видим:** два stage. Stage с `range` пишет shuffle (*shuffle write*), stage с `groupBy` его читает (*shuffle read*). Задачи примерно одинаковые (min ≈ med ≈ max), значит перекоса нет. Откройте ссылку на stage и найдите те же цифры в **Summary Metrics**.

---
# 2. PostgreSQL: чтение, pushdown и запись

### Как Spark читает таблицу по JDBC

```
❌ Без partitionColumn                           ✅ partitionColumn=order_id, numPartitions=8

 PostgreSQL                                       PostgreSQL
    │  SELECT * FROM orders                          │ … WHERE order_id <  1250001            → задача 1
    │  (1 соединение, 10 млн строк,                  │ … WHERE order_id >= 1250001 AND < …    → задача 2
    │   fetchsize=0 → весь результат в память)       │ …                                      → …
    ▼                                                │ … WHERE order_id >= 8750001            → задача 8
 [задача 1] ← всё делает одна задача,                ▼
              7 из 8 ядер простаивают             [з1][з2][з3][з4][з5][з6][з7][з8]   параллельно
```

| Опция | Зачем |
|---|---|
| `partitionColumn` | числовая / дата / timestamp колонка, **равномерно** распределённая, желательно с индексом |
| `lowerBound`, `upperBound` | только нарезают диапазон на куски, **данные не фильтруют** |
| `numPartitions` | = число параллельных соединений с БД. 8–32, а не 200: иначе положите базу |
| `fetchsize` | сколько строк забирать за раз. В драйвере PostgreSQL по умолчанию **0 = весь результат в память** |

**Где смотреть:** Stages → `Tasks: 1` у stage чтения, разброс `записей на задачу`, GC time. В PostgreSQL: `SELECT * FROM pg_stat_activity` покажет число соединений от Spark.

### 2.1 Чтение большой таблицы
#### ❌ Как НЕ надо: читать «в лоб»

In [ ]:
# ❌ КАК НЕ НАДО: одна партиция, fetchsize по умолчанию
orders_jdbc_bad = spark.read.jdbc(
    url=PG_URL,
    table="orders",
    properties={"user": PG_USER, "password": PG_PASSWORD},
)
print("Партиций (= параллельных задач):", orders_jdbc_bad.rdd.getNumPartitions())

with measure("JDBC ❌ 1 партиция", case="PostgreSQL: чтение 10 млн строк", variant="❌"):
    orders_jdbc_bad.write.format("noop").mode("overwrite").save()

#### ✅ Как надо: параллельное чтение по равномерному ключу + fetchsize
Границы берём запросом к БД, а не «на глаз».

In [ ]:
# ✅ КАК НАДО
jdbc_opts = {"url": PG_URL, "user": PG_USER, "password": PG_PASSWORD}

bounds = (spark.read.format("jdbc").options(**jdbc_opts)
          .option("query", "SELECT min(order_id) AS lo, max(order_id) AS hi FROM orders")
          .load().first())
print("Границы order_id:", bounds.lo, "…", bounds.hi)

orders_jdbc = (
    spark.read.format("jdbc").options(**jdbc_opts)
    .option("dbtable", "orders")
    .option("partitionColumn", "order_id")   # равномерный ключ с индексом (PK)
    .option("lowerBound", bounds.lo)
    .option("upperBound", bounds.hi + 1)
    .option("numPartitions", 8)              # 8 соединений = 8 ядер
    .option("fetchsize", 10_000)             # строки приходят пачками через курсор
    .load()
)
print("Партиций:", orders_jdbc.rdd.getNumPartitions())

with measure("JDBC ✅ 8 партиций по order_id + fetchsize", case="PostgreSQL: чтение 10 млн строк", variant="✅"):
    orders_jdbc.write.format("noop").mode("overwrite").save()

In [ ]:
# 📊 МЕТРИКИ: смотрим «задач», «задача max» и GC
show_metrics("JDBC ❌ 1 партиция", "JDBC ✅ 8 партиций по order_id + fetchsize")

**На что смотреть:** в ❌ у stage **1 задача**, она читает все 10 млн строк, а 7 ядер простаивают. В ✅ 8 задач примерно одинаковой длительности, по 1,25 млн строк каждая.

> ⚠️ `fetchsize=0` (значение по умолчанию в драйвере PostgreSQL) означает, что весь результат запроса материализуется в памяти задачи. На 10 млн строк у нас это ещё помещается в 6 ГБ, а на 100 млн будет `OutOfMemoryError`.

### 2.2 Перекошенный `partitionColumn`

```
partitionColumn = customer_id (1 … 2 000 000), numPartitions = 8
  задача 1: customer_id < 250 001  OR customer_id IS NULL  ← 60% (клиент №1) + 15% (NULL) + 3% ≈ 78% строк 😱
  задача 2: 250 001 … 499 999                              ← ~3%
  …
  задача 8: ≥ 1 750 001                                    ← ~3%
```

#### ❌ Как НЕ надо: партиционировать по перекошенной колонке

In [ ]:
# ❌ КАК НЕ НАДО: customer_id распределён неравномерно (горячий клиент + NULL)
orders_jdbc_skewed = (
    spark.read.format("jdbc").options(**jdbc_opts)
    .option("dbtable", "orders")
    .option("partitionColumn", "customer_id")
    .option("lowerBound", 1).option("upperBound", 2_000_001)
    .option("numPartitions", 8)
    .option("fetchsize", 10_000)
    .load()
)
with measure("JDBC ❌ partitionColumn=customer_id (перекос)", case="PostgreSQL: выбор partitionColumn", variant="❌"):
    orders_jdbc_skewed.write.format("noop").mode("overwrite").save()

#### ✅ Как надо: проверить распределение в БД **до** выбора колонки
Дешёвый запрос показывает, сколько строк попадёт в каждую из 8 партиций. Затем читаем по `order_id`, как в 2.1.

In [ ]:
# ✅ КАК НАДО: смотрим, как ляжут строки по 8 партициям для каждой колонки-кандидата
distribution = (spark.read.format("jdbc").options(**jdbc_opts).option("query", """
    SELECT 'customer_id' AS col, coalesce(width_bucket(customer_id, 1, 2000001, 8), 1) AS part, count(*) AS rows
    FROM orders GROUP BY 1, 2
    UNION ALL
    SELECT 'order_id', width_bucket(order_id, 1, 10000001, 8), count(*) FROM orders GROUP BY 1, 2
""").load().toPandas())
display(distribution.pivot(index="part", columns="col", values="rows").rename_axis("партиция"))

with measure("JDBC ✅ partitionColumn=order_id (равномерно)", case="PostgreSQL: выбор partitionColumn", variant="✅"):
    orders_jdbc.write.format("noop").mode("overwrite").save()

In [ ]:
# 📊 МЕТРИКИ: «записей на задачу min / med / max» и «задача max»
show_metrics("JDBC ❌ partitionColumn=customer_id (перекос)", "JDBC ✅ partitionColumn=order_id (равномерно)")

### 2.3 Pushdown: пусть фильтр и агрегацию делает база

```
❌ Фильтр, который Spark не умеет перевести в SQL:
   PostgreSQL ──── 10 млн строк по сети ────► Spark ── filter(year(...)) ──► 5 млн

✅ Фильтр переводится в WHERE:
   PostgreSQL ── WHERE created_at >= … ──► 5 млн строк ──► Spark

✅✅ Агрегация в базе (option "query"), если результат маленький:
   PostgreSQL ── GROUP BY status ──► 5 строк ──► Spark
```

**Где смотреть:** `explain()` → строка `PushedFilters: [...]` у `Scan JDBCRelation`; в метриках — `input записей` у stage чтения.

#### ❌ Как НЕ надо: фильтр через функцию над колонкой

In [ ]:
# ❌ КАК НЕ НАДО: year() не транслируется в SQL, поэтому PostgreSQL отдаёт все 10 млн строк
revenue_2025_bad = (orders_jdbc
    .filter(F.year("created_at") == 2025)
    .groupBy("status").agg(F.sum("amount").alias("revenue")))

explain_highlights(revenue_2025_bad, show_plan=False)
with measure("JDBC ❌ filter(year()) — без pushdown", case="PostgreSQL: pushdown", variant="❌"):
    revenue_2025_bad.collect()

#### ✅ Как надо: фильтр по диапазону (уходит в `WHERE`) или агрегация в БД

In [ ]:
# ✅ КАК НАДО (вариант 1): диапазон по самой колонке переводится в WHERE и попадает в PushedFilters
revenue_2025 = (orders_jdbc
    .filter((F.col("created_at") >= "2025-01-01") & (F.col("created_at") < "2026-01-01"))
    .groupBy("status").agg(F.sum("amount").alias("revenue")))

explain_highlights(revenue_2025, show_plan=False)
with measure("JDBC ✅ filter по диапазону — pushdown", case="PostgreSQL: pushdown", variant="✅"):
    revenue_2025.collect()

# ✅ КАК НАДО (вариант 2): агрегацию выполняет PostgreSQL, в Spark приходит 5 строк
revenue_2025_db = (spark.read.format("jdbc").options(**jdbc_opts).option("query", """
    SELECT status, sum(amount) AS revenue
    FROM orders
    WHERE created_at >= '2025-01-01' AND created_at < '2026-01-01'
    GROUP BY status
""").load())
with measure("JDBC ✅✅ агрегация в PostgreSQL (query)"):
    display(revenue_2025_db.toPandas())

In [ ]:
# 📊 МЕТРИКИ: сравните «input записей» у stage чтения
show_metrics("JDBC ❌ filter(year()) — без pushdown",
             "JDBC ✅ filter по диапазону — pushdown",
             "JDBC ✅✅ агрегация в PostgreSQL (query)")

### 2.4 Запись в PostgreSQL

```
❌ batchsize=1000, обычный JDBC batch:  INSERT …; INSERT …; INSERT …;   ← по оператору на строку
✅ reWriteBatchedInserts=true:          INSERT … VALUES (…), (…), (…)   ← драйвер склеивает пачку в один оператор
```

| Опция | Зачем |
|---|---|
| `?reWriteBatchedInserts=true` в URL | драйвер PostgreSQL склеивает batch в multi-row INSERT: обычно в 2–5 раз быстрее |
| `batchsize` | строк в одном batch (по умолчанию 1000) |
| `numPartitions` | максимум параллельных соединений на запись (Spark сделает `coalesce`) |
| `truncate=true` | при `mode("overwrite")` делать TRUNCATE вместо DROP + CREATE: сохраняются типы, индексы, права |

Источник для записи — агрегат «выручка по клиенту» (~1,4 млн строк). Закешируем его, чтобы замерять **только запись**.

In [ ]:
customer_revenue = (orders_jdbc
    .groupBy("customer_id")
    .agg(F.count("*").alias("orders_cnt"),
         F.sum("amount").alias("revenue"),
         F.max("created_at").alias("last_order_at"))
    .cache())
print("Строк к записи:", customer_revenue.count())

#### ❌ Как НЕ надо: запись с настройками по умолчанию

In [ ]:
# ❌ КАК НЕ НАДО: batchsize=1000, без reWriteBatchedInserts
with measure("JDBC запись ❌ по умолчанию", case="PostgreSQL: запись 1,4 млн строк", variant="❌"):
    (customer_revenue.write
        .mode("overwrite")
        .option("truncate", "true")
        .jdbc(PG_URL, "customer_revenue", properties={"user": PG_USER, "password": PG_PASSWORD}))

#### ✅ Как надо: пакетная вставка

In [ ]:
# ✅ КАК НАДО
with measure("JDBC запись ✅ reWriteBatchedInserts + batchsize", case="PostgreSQL: запись 1,4 млн строк", variant="✅"):
    (customer_revenue.write.format("jdbc")
        .option("url", PG_URL + "?reWriteBatchedInserts=true")
        .option("user", PG_USER).option("password", PG_PASSWORD)
        .option("dbtable", "customer_revenue")
        .option("batchsize", 10_000)
        .option("numPartitions", 8)
        .option("truncate", "true")
        .mode("overwrite")
        .save())

In [ ]:
# 📊 МЕТРИКИ
show_metrics("JDBC запись ❌ по умолчанию", "JDBC запись ✅ reWriteBatchedInserts + batchsize")
customer_revenue.unpersist()

---
# 3. Читаем план: `explain()`

Дальше работаем с данными, заранее выгруженными в **S3** в формате Parquet (подробно про S3 в разделе 6).

In [ ]:
orders    = spark.read.parquet(f"{S3_ROOT}/good/orders")      # 10 млн, partitionBy(order_month)
customers = spark.read.parquet(f"{S3_ROOT}/good/customers")   # 2 млн
products  = spark.read.parquet(f"{S3_ROOT}/good/products")    # 50 тыс.
countries = spark.read.parquet(f"{S3_ROOT}/good/countries")   # 200

orders.printSchema()

### Анатомия `explain("formatted")`

План **читается снизу вверх**: от чтения данных к результату. Каждый `Exchange` — это shuffle и граница stage.

```
== Physical Plan ==
AdaptiveSparkPlan (16)                 ← AQE включён. isFinalPlan=false: план ещё может поменяться при выполнении
+- HashAggregate (15)                  ← финальная агрегация
   +- Exchange (14)                    ← 🔴 SHUFFLE перед финальной агрегацией
      +- HashAggregate (13)            ← частичная агрегация ДО shuffle (map-side combine)
         +- Project (12)
            +- SortMergeJoin Inner (11)         ← 🟠 join через shuffle + сортировку обеих сторон
               :- Sort (5)
               :  +- Exchange (4)               ← 🔴 SHUFFLE левой таблицы: hashpartitioning(product_id, 200)
               :     +- Filter (3)
               :        +- ColumnarToRow (2)
               :           +- Scan parquet (1)  ← смотрим PushedFilters / PartitionFilters / ReadSchema
               +- Sort (10)
                  +- Exchange (9)               ← 🔴 SHUFFLE правой таблицы
                     +- …
                        +- Scan parquet (6)

(4) Exchange
Input [2]: [product_id#5, amount#7]
Arguments: hashpartitioning(product_id#5, 200), ENSURE_REQUIREMENTS, [plan_id=42]
           ^^^^^^^^^^^^^^^ по какому ключу и на сколько партиций перемешиваем
```

| Оператор | Что значит | Хорошо или плохо |
|---|---|---|
| `Exchange hashpartitioning(key, N)` | shuffle по ключу | дорого: диск + сеть |
| `Exchange rangepartitioning` | shuffle для `orderBy` | дорого + отдельный job на сэмплирование |
| `Exchange SinglePartition` | всё в одну партицию | 🔴 одна задача делает всю работу |
| `SortMergeJoin` | обе стороны: shuffle + sort | нормально для big ⋈ big |
| `BroadcastExchange` + `BroadcastHashJoin` | маленькая сторона копируется на все executors | 🟢 большая таблица не двигается |
| `PushedFilters: [...]` | фильтр отдан в источник | 🟢 читаем меньше |
| `PartitionFilters: [...]` | отсечение папок-партиций | 🟢 читаем меньше файлов |

### 3.1 Join с маленькой таблицей
#### ❌ Как НЕ надо: SortMergeJoin большой таблицы с маленьким справочником
Чтобы показать «плохой» статический план, выключаем AQE и автоматический broadcast. Именно так ведёт себя Spark, когда не знает размер таблицы (нет статистики, сложный подзапрос).

In [ ]:
# ❌ КАК НЕ НАДО: 10 млн заказов перемешиваются ради справочника на 50 тыс. строк
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false", "spark.sql.autoBroadcastJoinThreshold": "-1"})

by_category_smj = (orders
    .join(products, "product_id")
    .groupBy("category")
    .agg(F.sum("amount").alias("revenue")))

explain_highlights(by_category_smj)

with measure("Join ❌ SortMergeJoin со справочником", case="Join: большая ⋈ маленькая", variant="❌"):
    by_category_smj.collect()

**На что смотреть в плане:** два `Exchange hashpartitioning(product_id, 200)`, по одному на каждую сторону, затем `Sort` и `SortMergeJoin`. Все 10 млн заказов уходят в shuffle.

#### ✅ Как надо: broadcast маленькой таблицы

In [ ]:
# ✅ КАК НАДО: products копируется на каждый executor, orders не двигается
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false"})   # AQE выключен, чтобы было видно эффект именно broadcast

by_category_bhj = (orders
    .join(F.broadcast(products), "product_id")
    .groupBy("category")
    .agg(F.sum("amount").alias("revenue")))

explain_highlights(by_category_bhj)

with measure("Join ✅ BroadcastHashJoin", case="Join: большая ⋈ маленькая", variant="✅"):
    by_category_bhj.collect()
reset_conf()

In [ ]:
# 📊 МЕТРИКИ: сравните shuffle write и число stages
show_metrics("Join ❌ SortMergeJoin со справочником", "Join ✅ BroadcastHashJoin")

**Итог:** в ✅ нет `Exchange hashpartitioning` под join'ом, есть `BroadcastExchange` + `BroadcastHashJoin`. Shuffle write упал до размера частичных агрегатов.

> ⚠️ Broadcast — не галочка «сделать быстрее». Таблица собирается **на драйвере** и копируется **в память каждого executor**. Ориентир: до сотен МБ в памяти, а не на диске. Parquet на 100 МБ может развернуться в 1–2 ГБ. Лимит Spark — 8 ГБ, таймаут `spark.sql.broadcastTimeout` = 300 с. В `FULL OUTER JOIN` broadcast невозможен, в `LEFT JOIN` транслируется только правая сторона.

### 3.2 Join двух больших таблиц
Broadcast для `customers` (2 млн строк) не подходит. Если join по одному и тому же ключу повторяется постоянно, shuffle можно сделать **один раз при записи**: это **бакетирование** (bucketing).

```
❌ каждый запуск:   orders ──Exchange──┐                ✅ один раз при записи:  orders    → 16 бакетов по hash(customer_id)
                                     SortMergeJoin                              customers → 16 бакетов по hash(customer_id)
                 customers ─Exchange─┘                   каждый запуск: бакет i ⋈ бакет i → SortMergeJoin БЕЗ Exchange
```

#### ❌ Как НЕ надо: перемешивать обе большие таблицы при каждом запуске

In [ ]:
# ❌ КАК НЕ НАДО: оба Exchange при каждом запуске
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false", "spark.sql.autoBroadcastJoinThreshold": "-1"})

segment_revenue = (orders
    .join(customers, "customer_id")
    .groupBy("segment")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

explain_highlights(segment_revenue)
with measure("Big join ❌ shuffle обеих таблиц", case="Join: большая ⋈ большая", variant="❌"):
    segment_revenue.collect()

#### ✅ Как надо: бакетировать обе таблицы по ключу join'а (один раз)

In [ ]:
# Подготовка: пишем бакетированные таблицы один раз (это тоже shuffle, но только один раз)
import shutil
for table in ("orders_bucketed", "customers_bucketed"):
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    shutil.rmtree(f"/home/jovyan/spark-warehouse/{table}", ignore_errors=True)

with measure("Подготовка бакетов (однократно)"):
    (orders.repartition(16, "customer_id").write.mode("overwrite")
        .bucketBy(16, "customer_id").sortBy("customer_id").saveAsTable("orders_bucketed"))
    (customers.repartition(16, "customer_id").write.mode("overwrite")
        .bucketBy(16, "customer_id").sortBy("customer_id").saveAsTable("customers_bucketed"))

In [ ]:
# ✅ КАК НАДО: join бакетированных таблиц — в плане нет Exchange под SortMergeJoin
segment_revenue_bucketed = (spark.table("orders_bucketed")
    .join(spark.table("customers_bucketed"), "customer_id")
    .groupBy("segment")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

explain_highlights(segment_revenue_bucketed)
with measure("Big join ✅ бакетирование", case="Join: большая ⋈ большая", variant="✅"):
    segment_revenue_bucketed.collect()
reset_conf()

In [ ]:
# 📊 МЕТРИКИ
show_metrics("Big join ❌ shuffle обеих таблиц", "Big join ✅ бакетирование")

**На что смотреть:** в ✅ нет `Exchange` под `SortMergeJoin`, shuffle write почти нулевой. Но в обоих вариантах ⚖️ **одна задача сильно дольше остальных**: горячий клиент `customer_id = 1` целиком лежит в одном бакете. Shuffle убрали, перекос остался. Разберём его в разделе 5.

### 3.3 AQE: `explain()` до выполнения ≠ реальный план
Adaptive Query Execution пересобирает план **во время выполнения** по фактическим размерам данных. Пример: заказов дороже 9 990 всего ~0,1% (около 10 тыс.), но до выполнения Spark этого не знает: без статистики он оценивает размер по всем файлам `orders` и выбирает SortMergeJoin.

#### ❌ Как НЕ надо: верить `explain()`, когда включён AQE

In [ ]:
# ❌ Ошибка восприятия: explain() показывает НАЧАЛЬНЫЙ план (isFinalPlan=false)
reset_conf()
big_orders = (orders
    .filter(F.col("amount") > 9_990)                       # по факту ~10 тыс. строк из 10 млн
    .join(customers, "customer_id")
    .select("order_id", "full_name", "email", "amount"))

explain_highlights(big_orders, mode="simple")   # ← здесь SortMergeJoin

#### ✅ Как надо: смотреть финальный план после выполнения
В Spark UI это вкладка **SQL / DataFrame** → запрос (там всегда финальный план). В коде — `final_plan(df)` после действия.

In [ ]:
# ✅ Выполняем запрос и смотрим ФИНАЛЬНЫЙ план (isFinalPlan=true)
with measure("AQE: финальный план"):
    big_orders.collect()
final_plan(big_orders)    # ← AQE заменил SortMergeJoin на BroadcastHashJoin по факту размера

In [ ]:
# 📊 МЕТРИКИ: в операторах SQL виден реальный тип join'а и AQEShuffleRead
show_metrics("AQE: финальный план")

---
# 4. Оптимизация shuffle

```
 Stage N (map)                                     Stage N+1 (reduce)
 [задача 1]──┐ сериализация → локальный диск ┐  ┌──► [задача 1] забирает свой ключ со ВСЕХ map-задач
 [задача 2]──┼───────── Shuffle Write ───────┼──┼──► [задача 2]
 [задача 3]──┘                               └──┴──► [задача N]          Shuffle Read (сеть)

 Каждый Exchange = сериализация + запись на диск + передача по сети + новая граница stage
```

**Как бороться** (по убыванию эффекта):
1. **Избежать**: broadcast, bucketing (раздел 3).
2. **Уменьшить объём**: фильтр, выбор колонок, агрегация **до** shuffle.
3. **Правильное число партиций**: AQE coalesce вместо ручного `spark.sql.shuffle.partitions`.
4. **Не плодить лишние**: каждый `repartition`, `distinct`, `orderBy`, `groupBy` и окно добавляет Exchange. Проверяйте план.

### 4.1 Агрегировать до join'а
Задача: выручка и число заказов по каждому клиенту, с именем клиента.

#### ❌ Как НЕ надо: сначала join 10 млн строк, потом агрегация

In [ ]:
# ❌ КАК НЕ НАДО: в shuffle под join уходят все 8,5 млн заказов (без NULL)
reset_conf()
revenue_join_first = (orders
    .join(customers, "customer_id")
    .groupBy("customer_id", "full_name", "segment")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders_cnt")))

with measure("Shuffle ❌ join → groupBy", case="Shuffle: агрегация до join", variant="❌"):
    revenue_join_first.write.format("noop").mode("overwrite").save()

#### ✅ Как надо: сначала сжать факты агрегацией, потом join
Частичная агрегация (`HashAggregate` до `Exchange`) схлопывает строки прямо в map-задачах. 6 млн заказов горячего клиента превращаются в несколько строк ещё до shuffle, так что заодно уходит и перекос.

In [ ]:
# ✅ КАК НАДО
reset_conf()
per_customer = (orders
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders_cnt")))

revenue_agg_first = per_customer.join(customers.select("customer_id", "full_name", "segment"), "customer_id")

with measure("Shuffle ✅ groupBy → join", case="Shuffle: агрегация до join", variant="✅"):
    revenue_agg_first.write.format("noop").mode("overwrite").save()

In [ ]:
# 📊 МЕТРИКИ: shuffle write, «задача max», перекос
show_metrics("Shuffle ❌ join → groupBy", "Shuffle ✅ groupBy → join")

### 4.2 Число shuffle-партиций
`spark.sql.shuffle.partitions` (по умолчанию 200) — одно значение на все запросы: где-то партиций слишком много, где-то слишком мало. AQE склеивает мелкие партиции после shuffle (`AQEShuffleRead coalesced`).

```
❌ 2000 партиций на 1000 групп:   [·][·][·][·][·][·][·][·] … ×2000   ← задачи по миллисекунды, планирование дороже работы
✅ AQE coalesce:                   [██████]                           ← партиции склеены до ~64 МБ (advisoryPartitionSizeInBytes)
```

#### ❌ Как НЕ надо: ручное значение «с запасом» без AQE

In [ ]:
# ❌ КАК НЕ НАДО
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false", "spark.sql.shuffle.partitions": "2000"})

status_by_country = (orders
    .groupBy("country_id", "status")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("cnt")))

with measure("Партиции ❌ 2000 без AQE", case="Shuffle: число партиций", variant="❌"):
    status_by_country.write.format("noop").mode("overwrite").save()

#### ✅ Как надо: AQE coalesce

In [ ]:
# ✅ КАК НАДО: значение то же, но AQE само склеит партиции
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "true",
          "spark.sql.adaptive.coalescePartitions.enabled": "true",
          "spark.sql.shuffle.partitions": "2000"})

with measure("Партиции ✅ AQE coalesce", case="Shuffle: число партиций", variant="✅"):
    status_by_country.write.format("noop").mode("overwrite").save()
reset_conf()

In [ ]:
# 📊 МЕТРИКИ: число задач и «AQE: склеено партиций» у AQEShuffleRead
show_metrics("Партиции ❌ 2000 без AQE", "Партиции ✅ AQE coalesce")

---
# 5. Перекос данных (data skew)

### Как выглядит перекос в Spark UI

```
Stages → stage → Event Timeline
 задача 1   ██
 задача 2   ██
 задача 3   ███
 …
 задача 63  ██
 задача 64  ████████████████████████████████████████████  ← горячий ключ customer_id = 1
            └ 1 c ┘                                    └ stage ждёт одну задачу

Summary Metrics:  Duration   Min 0.1 s │ 25% 0.1 s │ Median 0.2 s │ 75% 0.2 s │ Max 12 s   ← Max ≫ Median
                  Shuffle Read Records  …            │ Median 50 K  │           │ Max 6 M    ← причина
```

**Решения:**
1. **AQE skew join**: Spark сам режет перекошенную партицию на части (только для join).
2. **Salting**: вручную размазываем горячий ключ по N «солёным» подключам.
3. **NULL-ключи**: размазываем случайным ключом или обрабатываем отдельно.
4. **Агрегировать до join / окна** (см. 4.1).

### 5.1 Диагностика: найти горячие ключи

In [ ]:
# Сначала цифры: какие ключи «горячие»
total = orders.count()
(orders.groupBy("customer_id").count()
    .withColumn("доля, %", F.round(F.col("count") / total * 100, 2))
    .orderBy(F.desc("count"))
    .show(5))

### 5.2 Join с горячим ключом
#### ❌ Как НЕ надо: join по перекошенному ключу без защиты

In [ ]:
# ❌ КАК НЕ НАДО: AQE выключен, 6 млн строк клиента №1 попадают в одну задачу
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false",
          "spark.sql.autoBroadcastJoinThreshold": "-1",
          "spark.sql.shuffle.partitions": "64"})

skewed_join = (orders
    .join(customers, "customer_id")
    .select("order_id", "customer_id", "segment", "amount",
            F.sha2(F.concat_ws("|", "order_id", "email", "amount"), 256).alias("row_hash"))  # «тяжёлая» обработка строк после join
)

with measure("Skew ❌ join без защиты", case="Skew: join с горячим ключом", variant="❌"):
    skewed_join.write.format("noop").mode("overwrite").save()

In [ ]:
# 📊 МЕТРИКИ: ищем ⚖️ — «задача max» в десятки раз больше медианы
show_metrics("Skew ❌ join без защиты")

#### ✅ Как надо (способ 1): AQE skew join
AQE считает партицию перекошенной, если она одновременно:
- больше `skewedPartitionFactor` × медиана (по умолчанию 5×);
- больше `skewedPartitionThresholdInBytes` (по умолчанию 256 МБ).

Перекошенная партиция режется на куски, а соответствующая партиция другой стороны **дублируется** к каждому куску.

> На наших учебных данных партиция весит меньше 256 МБ, поэтому снижаем порог. В проде обычно хватает значений по умолчанию.

In [ ]:
# ✅ КАК НАДО (способ 1)
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "true",
          "spark.sql.adaptive.skewJoin.enabled": "true",
          "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "3",
          "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "16MB",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes": "8MB",
          "spark.sql.autoBroadcastJoinThreshold": "-1",
          "spark.sql.shuffle.partitions": "64"})

with measure("Skew ✅ AQE skew join", case="Skew: join с горячим ключом", variant="✅ AQE"):
    skewed_join.write.format("noop").mode("overwrite").save()
reset_conf()

#### ✅ Как надо (способ 2): salting горячих ключей вручную
Нужен, когда AQE не помогает: агрегации и окна по ключу, очень сильный перекос, старый Spark.

```
orders (большая):       customer_id=1 → salt = rand(0..63)       → (1,0) (1,37) (1,5) …  6 млн строк размазаны по 64 ключам
customers (маленькая):  customer_id=1 → explode(salt 0..63)      → (1,0) (1,1) … (1,63) строка клиента ×64
остальные ключи:        salt = 0 с обеих сторон                   → ничего не меняется
join ON customer_id AND salt
```

In [ ]:
# ✅ КАК НАДО (способ 2): солим только горячие ключи из диагностики 5.1
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false",
          "spark.sql.autoBroadcastJoinThreshold": "-1",
          "spark.sql.shuffle.partitions": "64"})

SALT_BUCKETS = 64
HOT_KEYS = [1]
is_hot = F.col("customer_id").isin(HOT_KEYS)

orders_salted = orders.withColumn(
    "salt", F.when(is_hot, (F.rand(seed=42) * SALT_BUCKETS).cast("int")).otherwise(F.lit(0)))
customers_salted = customers.withColumn(
    "salt", F.explode(F.when(is_hot, F.sequence(F.lit(0), F.lit(SALT_BUCKETS - 1))).otherwise(F.array(F.lit(0)))))

salted_join = (orders_salted
    .join(customers_salted, ["customer_id", "salt"])
    .select("order_id", "customer_id", "segment", "amount",
            F.sha2(F.concat_ws("|", "order_id", "email", "amount"), 256).alias("row_hash")))

with measure("Skew ✅ salting", case="Skew: join с горячим ключом", variant="✅ salting"):
    salted_join.write.format("noop").mode("overwrite").save()
reset_conf()

In [ ]:
# 📊 МЕТРИКИ: сравните «задача max» и «записей на задачу max»; у AQE смотрите «AQE: перекошенных партиций»
show_metrics("Skew ❌ join без защиты", "Skew ✅ AQE skew join", "Skew ✅ salting")

### 5.3 NULL-ключи
Все `NULL` хешируются одинаково, поэтому в `LEFT JOIN` они **все попадают в одну партицию**. В `INNER JOIN` Spark сам добавляет фильтр `isnotnull(customer_id)` (видно в плане), а в `LEFT JOIN` так сделать нельзя: гостевые заказы должны остаться в результате.

Чтобы увидеть эффект именно от NULL, возьмём заказы без горячего клиента: 1,5 млн гостевых + 2,5 млн обычных.

#### ❌ Как НЕ надо: LEFT JOIN, в котором NULL-ключи шафлятся в одну задачу

In [ ]:
# ❌ КАК НЕ НАДО
reset_conf()
set_conf({"spark.sql.adaptive.enabled": "false",
          "spark.sql.autoBroadcastJoinThreshold": "-1",
          "spark.sql.shuffle.partitions": "64"})

orders_regular = orders.filter(F.col("customer_id").isNull() | (F.col("customer_id") != 1))
customers_slim = customers.select("customer_id", "segment", "email")

orders_enriched_bad = (orders_regular
    .join(customers_slim, "customer_id", "left")
    .select("order_id", "customer_id", "segment", "amount",
            F.sha2(F.concat_ws("|", "order_id", "email", "amount"), 256).alias("row_hash")))

with measure("NULL ❌ left join с NULL-ключами", case="Skew: NULL-ключи в left join", variant="❌"):
    orders_enriched_bad.write.format("noop").mode("overwrite").save()

#### ✅ Как надо: размазать NULL-ключи по всем задачам
NULL заменяем на **случайный отрицательный ключ**, которого точно нет в справочнике. Join для этих строк ничего не найдёт, как и с NULL, но строки разойдутся по всем задачам, а не лягут в одну.

```
❌ NULL, NULL, NULL, … (1,5 млн)     → hash(NULL) → одна задача
✅ -17, -503, -88, -941, …            → hash(-k)   → все 64 задачи, совпадений в customers нет
```

Альтернатива: `filter(isNull)` + `filter(isNotNull).join(...)` + `unionByName`. Но тогда источник читается дважды.

In [ ]:
# ✅ КАК НАДО: NULL → случайный отрицательный ключ (-1 … -1000), которого нет в customers
join_key = F.coalesce(F.col("customer_id"), -(F.floor(F.rand(seed=7) * 1000) + 1).cast("int"))

orders_enriched = (orders_regular
    .withColumn("join_key", join_key)
    .join(customers_slim.withColumnRenamed("customer_id", "join_key"), "join_key", "left")
    .select("order_id", "customer_id", "segment", "amount",
            F.sha2(F.concat_ws("|", "order_id", "email", "amount"), 256).alias("row_hash")))

with measure("NULL ✅ NULL размазаны по задачам", case="Skew: NULL-ключи в left join", variant="✅"):
    orders_enriched.write.format("noop").mode("overwrite").save()
reset_conf()

In [ ]:
# 📊 МЕТРИКИ
show_metrics("NULL ❌ left join с NULL-ключами", "NULL ✅ NULL размазаны по задачам")

### 5.4 Окно по перекошенному ключу
AQE skew join **не помогает** оконным функциям: `Window.partitionBy(customer_id)` обязан собрать все строки ключа в одной задаче и отсортировать их.

Задача: накопительная выручка клиента по дням.

#### ❌ Как НЕ надо: окно по сырым 10 млн строк

In [ ]:
# ❌ КАК НЕ НАДО: 6 млн строк клиента №1 сортируются в одной задаче
reset_conf()
w = Window.partitionBy("customer_id").orderBy("created_at")
running_bad = (orders
    .filter(F.col("customer_id").isNotNull())
    .withColumn("running_revenue", F.sum("amount").over(w)))

with measure("Window ❌ окно по сырым строкам", case="Skew: окно по горячему ключу", variant="❌"):
    running_bad.write.format("noop").mode("overwrite").save()

#### ✅ Как надо: сначала агрегировать до нужной гранулярности (день), потом окно
6 млн строк клиента №1 превращаются в ~730 строк (по одной на день).

In [ ]:
# ✅ КАК НАДО
reset_conf()
daily = (orders
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id", F.to_date("created_at").alias("order_date"))
    .agg(F.sum("amount").alias("day_revenue")))

w_daily = Window.partitionBy("customer_id").orderBy("order_date")
running_good = daily.withColumn("running_revenue", F.sum("day_revenue").over(w_daily))

with measure("Window ✅ агрегация → окно", case="Skew: окно по горячему ключу", variant="✅"):
    running_good.write.format("noop").mode("overwrite").save()

In [ ]:
# 📊 МЕТРИКИ: «задача max», spill, «записей на задачу max»
show_metrics("Window ❌ окно по сырым строкам", "Window ✅ агрегация → окно")

---
# 6. S3: подключение и оптимизация

### S3 — это не файловая система

```
 Локальный диск / HDFS                   S3 (объектное хранилище, HTTP API)
 ───────────────────────                 ───────────────────────────────────────────────
 ls     → метаданные, мгновенно          LIST   → HTTP-запрос, не больше 1000 ключей за раз, 50–200 мс
 open   → дёшево                         GET    → HTTP-запрос на КАЖДЫЙ файл (+ range-запросы к футеру Parquet)
 rename → атомарно, мгновенно            rename → НЕТ: COPY каждого файла + DELETE (медленно, не атомарно)
 папки  → есть                           папки  → нет, это просто префиксы ключей «a/b/c.parquet»
```

**Отсюда три правила:**
1. **Колоночный формат** (Parquet / ORC): читаем только нужные колонки и row-group'ы.
2. **Мало крупных файлов** (128 МБ – 1 ГБ) вместо тысяч мелких: каждый файл — это HTTP-запросы.
3. **Committer без rename** (magic / S3A committers): иначе commit = копирование всего результата.

### Где смотреть, что всё плохо

```
SQL / DataFrame → запрос → блок «Scan parquet»
 ┌──────────────────────────────────────────────┐
 │ Scan parquet                                 │
 │  number of files read:      2 920   ← 🔴 тысячи файлов
 │  size of files read:        280 MB  ← 🔴 в среднем ~100 КБ на файл
 │  number of partitions read: 730     │
 └──────────────────────────────────────────────┘
Jobs → Event Timeline:  ▓▓ job ▓▓ ……… 20 c пусто ……… ▓▓ job ▓▓   ← 🔴 драйвер листит или переименовывает файлы
Stages → Input Size:    1.2 GB при нужных 2 колонках            ← 🔴 читаем лишнее (CSV, select *)
```

В ячейках «📊 Метрики» есть строка **☁️ S3** — это счётчики самого коннектора S3A: сколько байт реально прочитано по сети и сколько HTTP-запросов (LIST / HEAD / GET / PUT / COPY / DELETE) ушло в хранилище.

### Подключение
Ключевые настройки — в ячейке SparkSession (раздел 0): `endpoint`, `path.style.access`, ключи, пул соединений. Посмотрим, что лежит в бакете.

In [ ]:
# Что лежит в S3: файлы, папки, средний размер файла
datasets = ["good/orders", "good/customers", "good/products", "good/countries",
            "raw/orders_csv", "bad/orders_small_files"]
display(pd.DataFrame([s3_stats(f"{S3_ROOT}/{d}") for d in datasets]).set_index("путь"))

### 6.1 Формат файлов: CSV против Parquet

```
CSV (строчный):     читаем ВСЕ байты всех строк → парсим текст → определяем типы
Parquet (колоночный): footer → нужные колонки → нужные row-group'ы (min/max статистика)
                    ┌ row group 1 ┐┌ row group 2 ┐
                    │ amount  ███ ││ amount  ███ │  ← читаем только amount и status
                    │ status  ███ ││ status  ███ │
                    │ email   ░░░ ││ email   ░░░ │  ← не читаем
                    └─────────────┘└─────────────┘
```

#### ❌ Как НЕ надо: CSV + `inferSchema`
`inferSchema=true` — это **отдельный проход по всем данным** только ради определения типов.

In [ ]:
# ❌ КАК НЕ НАДО
with measure("S3 ❌ CSV + inferSchema", case="S3: формат файлов", variant="❌"):
    orders_csv = (spark.read
        .option("header", True)
        .option("inferSchema", True)          # ← полный лишний проход по S3
        .csv(f"{S3_ROOT}/raw/orders_csv"))
    status_revenue_csv = orders_csv.groupBy("status").agg(F.sum("amount").alias("revenue")).collect()

#### ✅ Как надо: Parquet (схема в футере, читаются только нужные колонки)
Если источник обязан быть CSV, задавайте схему явно: `.schema("order_id BIGINT, ...")`, без `inferSchema`.

In [ ]:
# ✅ КАК НАДО
with measure("S3 ✅ Parquet", case="S3: формат файлов", variant="✅"):
    status_revenue_pq = orders.groupBy("status").agg(F.sum("amount").alias("revenue")).collect()

explain_highlights(orders.groupBy("status").agg(F.sum("amount")), show_plan=False)   # ← ReadSchema: только 2 колонки

In [ ]:
# 📊 МЕТРИКИ: «☁️ S3 прочитано» и число stages (у CSV + inferSchema есть лишний проход)
show_metrics("S3 ❌ CSV + inferSchema", "S3 ✅ Parquet")

### 6.2 Partition pruning и predicate pushdown

```
good/orders/
 ├─ order_month=2024-01/part-0000.parquet
 ├─ order_month=2024-02/part-0000.parquet
 ├─ …                                          ❌ filter(year(created_at)==2025 & month(...)==3) → читаем ВСЕ 24 папки
 └─ order_month=2025-12/part-0000.parquet      ✅ filter(order_month == "2025-03")              → читаем 1 папку
```

**Где смотреть:** `explain()` → `PartitionFilters: [...]` (пусто = pruning не сработал); SQL → Scan → `number of files read`, `size of files read`.

#### ❌ Как НЕ надо: фильтр через функцию над колонкой, по которой НЕ партиционированы данные

In [ ]:
# ❌ КАК НЕ НАДО
march_bad = (orders
    .filter((F.year("created_at") == 2025) & (F.month("created_at") == 3))
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

explain_highlights(march_bad, show_plan=False)
with measure("S3 ❌ фильтр без partition pruning", case="S3: partition pruning", variant="❌"):
    march_bad.collect()

#### ✅ Как надо: фильтр по колонке партиционирования

In [ ]:
# ✅ КАК НАДО
march_good = (orders
    .filter(F.col("order_month") == "2025-03")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

explain_highlights(march_good, show_plan=False)
with measure("S3 ✅ partition pruning", case="S3: partition pruning", variant="✅"):
    march_good.collect()

In [ ]:
# 📊 МЕТРИКИ: «файлов прочитано» и «объём файлов» у Scan parquet
show_metrics("S3 ❌ фильтр без partition pruning", "S3 ✅ partition pruning")

### 6.3 Проблема мелких файлов: чтение

```
❌ bad/orders_small_files/                         ✅ good/orders/
   order_date=2024-01-01/ part-0..3 (по ~100 КБ)      order_month=2024-01/ part-0 (~10 МБ)
   order_date=2024-01-02/ part-0..3                   …
   … 730 папок × 4 файла = 2 920 файлов               24 папки × 1 файл
   → LIST по 730 папкам + 2 920 × (HEAD + GET footer + GET data)   → 24 × (HEAD + GET)
```

#### ❌ Как НЕ надо: тысячи мелких файлов

In [ ]:
# ❌ КАК НЕ НАДО: те же 10 млн строк, разложенные по 2 920 мелким файлам
with measure("S3 ❌ мелкие файлы (2 920 шт.)", case="S3: чтение мелких файлов", variant="❌"):
    orders_small = spark.read.parquet(f"{S3_ROOT}/bad/orders_small_files")   # ← листинг 730 папок
    orders_small.groupBy("status").agg(F.sum("amount").alias("revenue")).collect()

#### ✅ Как надо: несколько крупных файлов

In [ ]:
# ✅ КАК НАДО
with measure("S3 ✅ крупные файлы (24 шт.)", case="S3: чтение мелких файлов", variant="✅"):
    orders_big = spark.read.parquet(f"{S3_ROOT}/good/orders")
    orders_big.groupBy("status").agg(F.sum("amount").alias("revenue")).collect()

In [ ]:
# 📊 МЕТРИКИ: «файлов прочитано», число HTTP-запросов к S3, время задач
show_metrics("S3 ❌ мелкие файлы (2 920 шт.)", "S3 ✅ крупные файлы (24 шт.)")

### 6.4 Committer: запись без rename

```
❌ FileOutputCommitter (по умолчанию):
   задача пишет  →  _temporary/attempt_X/part-0.parquet
   commit задачи →  rename в _temporary/task_X/        = COPY + DELETE каждого файла
   commit job'а  →  rename в итоговую папку (драйвер)  = ещё раз COPY + DELETE каждого файла
   + десятки HEAD/LIST-запросов на проверку «папок», которых в S3 нет

✅ S3A magic committer:
   задача пишет  →  multipart upload сразу в итоговый ключ, но НЕ завершает его
   commit job'а  →  драйвер завершает все multipart upload'ы  (без копирования данных)
```

Нужен модуль `spark-hadoop-cloud` и три настройки. Их можно включать на уровне сессии.

> 💡 **Spark 4.x** при наличии `spark-hadoop-cloud` в classpath **сам включает magic committer** для S3A. В Spark 3.x и в сборках без модуля по умолчанию работает FileOutputCommitter, то есть rename. Проверить, что у вас: Spark UI → **Environment** → `spark.sql.sources.commitProtocolClass`, или ячейка ниже.

> На тестовом прогоне при подготовке вебинара запись ~1 000 файлов через FileOutputCommitter заняла **361 c** (1 984 COPY и 9 127 HEAD-запросов), а через magic committer — **31 c**. Здесь пишем всего 31 файл, чтобы демо было коротким.

In [ ]:
MAGIC_COMMITTER = {
    "spark.sql.sources.commitProtocolClass": "org.apache.spark.internal.io.cloud.PathOutputCommitProtocol",
    "spark.sql.parquet.output.committer.class": "org.apache.spark.internal.io.cloud.BindingParquetOutputCommitter",
    "fs.s3a.committer.name": "magic",
}
# Классический путь: так пишут Spark 3.x и сборки без spark-hadoop-cloud
CLASSIC_COMMITTER = {
    "spark.sql.sources.commitProtocolClass": "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol",
    "spark.sql.parquet.output.committer.class": "org.apache.parquet.hadoop.ParquetOutputCommitter",
    "fs.s3a.committer.name": "file",
}

print("Сейчас в сессии:")
for key in MAGIC_COMMITTER:
    print(f"  {key} = {spark.conf.get(key, spark.conf.get('spark.hadoop.' + key, None))}")

# Один месяц заказов (~420 тыс. строк), раскладка по дням: 31 папка × 1 файл
march = (orders.filter(F.col("order_month") == "2025-03")
         .withColumn("order_date", F.to_date("created_at"))
         .drop("order_month"))
TMP_V1, TMP_MAGIC = f"{S3_ROOT}/tmp/commit_file_v1", f"{S3_ROOT}/tmp/commit_magic"


def committer_of(path):
    """Какой committer записал данные: magic committer пишет JSON в _SUCCESS, классический — пустой файл."""
    fs, p = _fs(f"{path}/_SUCCESS")
    stream = fs.open(p)
    text = spark._jvm.org.apache.commons.io.IOUtils.toString(stream, "UTF-8")
    stream.close()
    return json.loads(text)["committer"] if text.strip() else "FileOutputCommitter (rename)"

#### ❌ Как НЕ надо: FileOutputCommitter (rename) поверх S3

In [ ]:
# ❌ КАК НЕ НАДО: классический FileOutputCommitter (по умолчанию в Spark 3.x)
reset_conf()
set_conf(CLASSIC_COMMITTER)
with measure("S3 commit ❌ FileOutputCommitter (rename)", case="S3: committer", variant="❌"):
    march.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(TMP_V1)
print("committer:", committer_of(TMP_V1))
reset_conf()

#### ✅ Как надо: magic committer

In [ ]:
# ✅ КАК НАДО
reset_conf()
set_conf(MAGIC_COMMITTER)
with measure("S3 commit ✅ magic committer", case="S3: committer", variant="✅"):
    march.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(TMP_MAGIC)
print("committer:", committer_of(TMP_MAGIC))

In [ ]:
# 📊 МЕТРИКИ: задачи одинаковые; смотрите строку ☁️ S3 (COPY / HEAD / LIST) и время ВНЕ задач (commit на драйвере)
show_metrics("S3 commit ❌ FileOutputCommitter (rename)", "S3 commit ✅ magic committer")

### 6.5 Проблема мелких файлов: запись

```
❌ df.repartition(32).write.partitionBy("order_date")
   каждая из 32 задач пишет кусочек в КАЖДУЮ из 31 папки → 32 × 31 ≈ 1 000 файлов по ~10 КБ

✅ df.repartition("order_date").write.partitionBy("order_date")
   все строки одной даты попадают в одну задачу → 1 файл на папку (31 файл)
   (для крупных партиций добавьте option("maxRecordsPerFile", N))
```
Оба варианта пишем уже через magic committer: даже с ним тысяча мелких файлов — это тысяча PUT-запросов и медленное чтение потом.

In [ ]:
TMP_BAD, TMP_GOOD = f"{S3_ROOT}/tmp/write_small_files", f"{S3_ROOT}/tmp/write_compacted"
reset_conf()
set_conf(MAGIC_COMMITTER)

#### ❌ Как НЕ надо: `repartition(N)` + `partitionBy` по другой колонке

In [ ]:
# ❌ КАК НЕ НАДО
with measure("S3 запись ❌ мелкие файлы", case="S3: запись (раскладка файлов)", variant="❌"):
    march.repartition(32).write.mode("overwrite").partitionBy("order_date").parquet(TMP_BAD)
display(pd.DataFrame([s3_stats(TMP_BAD)]))

#### ✅ Как надо: `repartition` по колонке партиционирования

In [ ]:
# ✅ КАК НАДО
with measure("S3 запись ✅ 1 файл на партицию", case="S3: запись (раскладка файлов)", variant="✅"):
    march.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(TMP_GOOD)
display(pd.DataFrame([s3_stats(TMP_GOOD)]))
reset_conf()

In [ ]:
# 📊 МЕТРИКИ: число PUT-запросов, файлов и время
show_metrics("S3 запись ❌ мелкие файлы", "S3 запись ✅ 1 файл на партицию")

### 6.6 Шпаргалка по настройкам S3A

| Настройка | Значение | Зачем |
|---|---|---|
| `fs.s3a.endpoint` | `https://s3.twcstorage.ru` | S3-совместимое хранилище (не AWS) |
| `fs.s3a.path.style.access` | `true` | адрес `host/bucket/key` вместо `bucket.host/key` |
| `fs.s3a.connection.maximum` | 100–500 | пул HTTP-соединений: при параллельном чтении упираются именно в него |
| `fs.s3a.threads.max` | 32–128 | потоки для загрузки и листинга |
| `fs.s3a.committer.name` | `magic` | commit без rename (+ `commitProtocolClass`, см. 6.5) |
| `fs.s3a.fast.upload.buffer` | `disk` / `bytebuffer` | буфер multipart upload: `bytebuffer` быстрее, но ест off-heap память |
| `fs.s3a.multipart.size` | `64M`–`128M` | размер части multipart upload |
| `fs.s3a.experimental.input.fadvise` | `random` | для Parquet/ORC: range-запросы вместо чтения файла целиком |
| `spark.sql.files.maxPartitionBytes` | `128MB`–`512MB` | сколько байт файлов уходит в одну задачу чтения |
| `spark.sql.files.openCostInBytes` | `4MB` | «стоимость» открытия файла: Spark склеивает мелкие файлы в одну задачу |
| `spark.sql.sources.parallelPartitionDiscovery.threshold` | `32` | выше этого числа путей листинг идёт отдельным Spark job'ом |

**Ещё про S3:**
- `mode("overwrite")` с `partitionBy` по умолчанию удаляет **все** партиции. Для перезаписи только затронутых: `spark.sql.sources.partitionOverwriteMode=dynamic`.
- Для `MERGE`, `UPDATE`, time travel и компакции мелких файлов используют табличные форматы **Delta Lake / Apache Iceberg** поверх S3.

In [ ]:
# Уборка временных файлов демо в S3
for path in [TMP_BAD, TMP_GOOD, TMP_V1, TMP_MAGIC]:
    s3_delete(path)
print("tmp очищен")

---
# 7. Итоги: «было → стало»

In [ ]:
# Сводная таблица по всем парам ❌ / ✅
rows = []
for label, run in RUNS.items():
    if run["case"]:
        rows.append({"кейс": run["case"], "вариант": run["variant"], "запуск": label, "время, c": round(run["wall"], 1)})
results = pd.DataFrame(rows)

summary = []
for case, grp in results.groupby("кейс", sort=False):
    bad = grp[grp["вариант"].str.startswith("❌")]["время, c"].min()
    good = grp[grp["вариант"].str.startswith("✅")]["время, c"].min()
    summary.append({"кейс": case, "❌ было, c": bad, "✅ стало, c": good,
                    "ускорение": f"×{bad / good:.1f}" if good else ""})
summary = pd.DataFrame(summary).set_index("кейс")
display(summary)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 0.55 * len(summary) + 1))
y = range(len(summary))
ax.barh([i + 0.2 for i in y], summary["❌ было, c"], height=0.4, label="❌ было", color="#d9534f")
ax.barh([i - 0.2 for i in y], summary["✅ стало, c"], height=0.4, label="✅ стало", color="#5cb85c")
ax.set_yticks(list(y))
ax.set_yticklabels(summary.index)
ax.invert_yaxis()
ax.set_xlabel("секунды")
ax.set_title("Оптимизация PySpark: было → стало")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### ✅ Чеклист оптимизации PySpark

**Чтение из БД (JDBC)**
- [ ] `partitionColumn` по **равномерному** ключу + `lowerBound`/`upperBound` из `min/max` + `numPartitions` 8–32
- [ ] `fetchsize` 10 000+ (в драйвере PostgreSQL по умолчанию весь результат грузится в память)
- [ ] фильтры, которые проталкиваются в источник (`PushedFilters`), а тяжёлые агрегации — через `query`
- [ ] запись: `reWriteBatchedInserts=true`, `batchsize`, `truncate=true`

**Join и shuffle**
- [ ] маленькая таблица → `broadcast` (до сотен МБ в памяти)
- [ ] большая ⋈ большая по одному ключу постоянно → bucketing
- [ ] фильтр, выбор колонок и **агрегация до join'а**
- [ ] AQE включён (`spark.sql.adaptive.enabled=true`), финальный план смотрим в Spark UI → SQL
- [ ] каждый `Exchange` в плане должен быть осознанным

**Перекос**
- [ ] в Spark UI: Stages → Summary Metrics, **Max ≫ Median**
- [ ] найти горячие ключи через `groupBy(key).count()`
- [ ] AQE skew join → salting → NULL-ключи размазать → агрегация до окна

**S3**
- [ ] Parquet / ORC, а не CSV/JSON; никакого `inferSchema` в проде
- [ ] партиционирование по колонке частых фильтров, проверка `PartitionFilters` в плане
- [ ] файлы 128 МБ – 1 ГБ: `repartition(колонка партиций)` перед записью, компакция
- [ ] magic committer вместо rename (в Spark 4 включается сам при наличии `spark-hadoop-cloud`; проверьте Environment)
- [ ] пул соединений S3A (`connection.maximum`, `threads.max`)

**Измерение**
- [ ] замер через `write.format("noop")`, а не `count()`
- [ ] Spark UI: Stages (Summary Metrics, Spill), SQL (метрики операторов), Jobs (Event Timeline)
- [ ] History Server, чтобы разбирать запуски после их завершения